# 01 — Dataset Generation (Batch)
Generate game-specific questions from PDFs using vLLM on the A100 server.

**Run on:** A100 server (after SSH tunnel is active)  
**Model:** `Qwen/Qwen2.5-7B-Instruct-AWQ` (10 GB MIG slice)

> Acknowledgement: Computational resources supported by KU Nontri AI, Kasetsart University

## Quick Start
1. Start vLLM on server: `sbatch ~/ku_prep_arena/ai/scripts/slurm_vllm.sh`
2. Open SSH tunnel (local machine): `ssh -N -L 8000:dgx-XX:8000 aip04@br1.paas.ku.ac.th`
3. Run this notebook cell by cell

In [9]:
import os, json, sys, re
from pathlib import Path
from openai import OpenAI
from tqdm.notebook import tqdm

# ── Connect to vLLM ──────────────────────────────────────────────────────────
# LOCAL machine (after tunnel):   AI_BASE_URL=http://localhost:8001/v1
# On server (br2, no tunnel):     AI_BASE_URL=http://dgx-02:8001/v1
BASE_URL = os.getenv("AI_BASE_URL", "http://localhost:8001/v1")
API_KEY  = os.getenv("AI_API_KEY",  "none")
MODEL    = os.getenv("AI_MODEL",    "Qwen/Qwen2.5-7B-Instruct-AWQ")

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

# Quick health check
try:
    models = client.models.list()
    print(f"✅ Connected to: {BASE_URL}")
    print(f"✅ Available model: {models.data[0].id if models.data else 'none'}")
except Exception as e:
    print(f"❌ Cannot connect: {e}")
    print("   → LOCAL: ssh -N -L 8001:dgx-02:8001 aip04@br1.paas.ku.ac.th")
    print("   → SERVER: AI_BASE_URL=http://dgx-02:8001/v1 แล้วรัน cell ใหม่")
    raise

✅ Connected to: http://localhost:8001/v1
✅ Available model: Qwen/Qwen2.5-7B-Instruct-AWQ


## Config

In [10]:
# ── Config ─────────────────────────────────────────────────────────────────
PDF_DIR   = Path("../dataset/sample_pdfs")
OUT_DIR   = Path("../dataset/generated")
GAME_TYPES = ["flappy", "racer", "shooter", "snake", "bricks"]

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"PDFs   : {PDF_DIR}")
print(f"Output : {OUT_DIR}")

PDFs   : ../dataset/sample_pdfs
Output : ../dataset/generated


## Helper functions

In [11]:
import random

NO_CHINESE = (
    "CRITICAL: Do NOT output Chinese (中文), Japanese, Korean, Arabic, or any script "
    "not present in the document. Write ONLY in the same language as the document "
    "(Thai = ภาษาไทย, English = English)."
)

GAME_HINTS = {
    "flappy":  "Keep questions SHORT (under 12 words) and choices very brief (1-4 words). Player reads while flying.",
    "racer":   "Keep EVERY question under 12 words. Choices must be 1-4 words each — no long phrases.",
    "shooter": "Focus on IDENTIFICATION questions: 'Which term describes…?'. Include plausible distractors.",
    "snake":   "Prefer SEQUENTIAL questions: 'What is the FIRST step in…?'. Choices = different stages.",
    "bricks":  "Focus on DEFINITIONS: 'What does X mean?'. Test precise vocabulary.",
}

DIFFICULTY_GUIDE = """
Difficulty guide (MUST follow the exact distribution: 3 easy, 5 medium, 2 hard):
- difficulty 1 (easy)   — direct recall: "What is X?" / "Which formula is…?" — 3 questions
- difficulty 2 (medium) — apply/compare: requires understanding, not just recall — 5 questions
- difficulty 3 (hard)   — analyze/synthesize: combine 2+ concepts, evaluate trade-offs,
                          "Given that X, what happens if Y?" or "Which scenario causes Z?" — 2 questions
"""

def extract_text_simple(pdf_path: Path, max_chars: int = 2_500) -> str:
    try:
        import pdfplumber
        with pdfplumber.open(pdf_path) as pdf:
            pages = [p.extract_text() or "" for p in pdf.pages]
        return "\n".join(pages)[:max_chars]
    except ImportError:
        try:
            from pypdf import PdfReader
            reader = PdfReader(str(pdf_path))
            text = "\n".join(p.extract_text() or "" for p in reader.pages)
            return text[:max_chars]
        except ImportError:
            raise ImportError("Install pdfplumber or pypdf: pip install pdfplumber")

def extract_json_array(raw: str) -> str:
    """Extract first complete JSON array using bracket balancing."""
    start = raw.find('[')
    if start == -1:
        raise ValueError(f"No '[' found in response:\n{raw[:200]}")
    depth, in_string, escape_next = 0, False, False
    for i, ch in enumerate(raw[start:], start):
        if escape_next:
            escape_next = False
            continue
        if ch == '\\' and in_string:
            escape_next = True
            continue
        if ch == '"':
            in_string = not in_string
        elif not in_string:
            if ch == '[':
                depth += 1
            elif ch == ']':
                depth -= 1
                if depth == 0:
                    return raw[start:i + 1]
    raise ValueError("Unbalanced brackets — JSON array not complete")

def shuffle_answers(questions: list[dict]) -> list[dict]:
    """Stratified shuffle: guarantees A/B/C/D are balanced across the question set."""
    n = len(questions)
    # Build a balanced target position list: [0,1,2,3,0,1,2,3,...] then shuffle
    positions = [i % 4 for i in range(n)]
    random.shuffle(positions)

    for q, target_pos in zip(questions, positions):
        choices = q.get("choices", [])
        correct_idx = q.get("correct", 0)
        if not (0 <= correct_idx < len(choices)):
            correct_idx = 0
        correct_text = choices[correct_idx]
        others = [c for j, c in enumerate(choices) if j != correct_idx]
        random.shuffle(others)
        # Place correct answer at target_pos, fill rest with distractors
        new_choices = others[:target_pos] + [correct_text] + others[target_pos:3]
        q["choices"] = new_choices  # exactly 4 items
        q["correct"] = target_pos
    return questions

def build_messages(text: str, game_type: str) -> list[dict]:
    hint = GAME_HINTS.get(game_type, "")
    system = f"{hint} {NO_CHINESE}\n\nRespond ONLY with a valid JSON array — no markdown, no explanation."
    user = f"""Create exactly 10 multiple-choice questions from the study material below.

CRITICAL RULES:
- Write ONLY in the SAME language as the document (Thai or English)
- Do NOT use Chinese, Japanese, Korean, or any other language
- Each question must have exactly 4 choices
- Only 1 correct answer per question
- Include a brief explanation
{DIFFICULTY_GUIDE}
Return ONLY this JSON format:
[{{"id":1,"question":"...","choices":["A","B","C","D"],"correct":0,"explanation":"...","difficulty":2}}]

Study Material:
{text}"""
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]

def generate_questions(text: str, game_type: str) -> list[dict]:
    messages = build_messages(text, game_type)
    res = client.chat.completions.create(
        model=MODEL, messages=messages, temperature=0.5, max_tokens=1_400
    )
    raw = res.choices[0].message.content or "[]"
    questions = json.loads(extract_json_array(raw))
    return shuffle_answers(questions)

print("✅ Functions defined")

✅ Functions defined


## Generate questions for all PDFs × all game types

In [ ]:
pdf_files = sorted(PDF_DIR.glob("*.pdf"))
print(f"Found {len(pdf_files)} PDFs: {[p.name for p in pdf_files]}")

results = []
for pdf_path in pdf_files:
    print(f"\n── {pdf_path.name}")
    text = extract_text_simple(pdf_path)  # fixed: was extract_text()
    print(f"   Extracted {len(text)} chars")

    for game_type in tqdm(GAME_TYPES, desc="  game types"):
        out_file = OUT_DIR / f"{pdf_path.stem}_{game_type}.json"
        if out_file.exists():
            print(f"   [SKIP] {out_file.name} already exists")
            continue
        try:
            questions = generate_questions(text, game_type)  # fixed: was call_llm()
            for i, q in enumerate(questions):
                q["game_type"]  = game_type
                q["source_doc"] = pdf_path.name
                q.setdefault("id", i + 1)
            out_file.write_text(json.dumps(questions, ensure_ascii=False, indent=2), encoding="utf-8")
            print(f"   ✓ {out_file.name}  ({len(questions)} questions)")
            results.append({"file": out_file.name, "n": len(questions), "ok": True})
        except Exception as e:
            print(f"   ✗ {game_type}: {e}")
            results.append({"file": f"{pdf_path.stem}_{game_type}.json", "error": str(e), "ok": False})

print(f"\nDone: {sum(r['ok'] for r in results)}/{len(results)} succeeded")

Found 3 PDFs: ['670E74B8BEC523762D0AC71DAD2B48B65BA6A497_232m0077 fund chem midterm_compressed.pdf', 'Calculus_1_Midterm_สำเนา_compressed.pdf', 'python101_workbook_v1.0.2_compressed.pdf']

── 670E74B8BEC523762D0AC71DAD2B48B65BA6A497_232m0077 fund chem midterm_compressed.pdf
   Extracted 2500 chars


  game types:   0%|          | 0/5 [00:00<?, ?it/s]

   ✓ 670E74B8BEC523762D0AC71DAD2B48B65BA6A497_232m0077 fund chem midterm_compressed_flappy.json  (10 questions)
   ✓ 670E74B8BEC523762D0AC71DAD2B48B65BA6A497_232m0077 fund chem midterm_compressed_racer.json  (10 questions)
   ✓ 670E74B8BEC523762D0AC71DAD2B48B65BA6A497_232m0077 fund chem midterm_compressed_shooter.json  (10 questions)
   ✓ 670E74B8BEC523762D0AC71DAD2B48B65BA6A497_232m0077 fund chem midterm_compressed_snake.json  (10 questions)
   ✓ 670E74B8BEC523762D0AC71DAD2B48B65BA6A497_232m0077 fund chem midterm_compressed_bricks.json  (10 questions)

── Calculus_1_Midterm_สำเนา_compressed.pdf
   Extracted 2500 chars


  game types:   0%|          | 0/5 [00:00<?, ?it/s]

   ✗ flappy: Invalid \escape: line 2 column 44 (char 45)
   ✗ racer: Unbalanced brackets — JSON array not complete
   ✓ Calculus_1_Midterm_สำเนา_compressed_shooter.json  (10 questions)
   ✗ snake: Invalid \escape: line 2 column 57 (char 58)
   ✗ bricks: Invalid \escape: line 4 column 84 (char 857)

── python101_workbook_v1.0.2_compressed.pdf
   Extracted 2500 chars


  game types:   0%|          | 0/5 [00:00<?, ?it/s]

   ✓ python101_workbook_v1.0.2_compressed_flappy.json  (10 questions)
   ✓ python101_workbook_v1.0.2_compressed_racer.json  (10 questions)


In [ ]:
# ── Patch existing files: add difficulty=2 where missing ─────────────────────
patched_files = 0
for f in OUT_DIR.glob("*.json"):
    qs = json.loads(f.read_text(encoding="utf-8"))
    changed = False
    for q in qs:
        if "difficulty" not in q:
            q["difficulty"] = 2   # default medium
            changed = True
    if changed:
        f.write_text(json.dumps(qs, ensure_ascii=False, indent=2), encoding="utf-8")
        print(f"  Patched: {f.name}")
        patched_files += 1

print(f"\nDone: {patched_files} files patched")